# Configure Dataset and Spark matching with hypex.config

[All How-to Guides](README.md)

`DatasetConfig` controls display and limits some operations that collect data to the driver. `MatchingConfig` controls distributed neighbor search. HypEx reads settings from the classes in the current Python process. Use `patch.multiple()` to override settings for a block. Exiting the block restores the previous values even if an exception occurs.

For the Spark sections, set `JAVA_HOME` to Java 17.

## Shorten the Dataset preview

Set `DISPLAY_ROWS=2` and `DISPLAY_COLS=1`. When HypEx truncates a long or wide preview, these settings determine how many rows and columns it shows at each edge. Check the full dimensions with `shape` afterward.

In [1]:
from unittest.mock import patch

from IPython.display import display

from hypex.config import DatasetConfig, MatchingConfig
from hypex.dataset import Dataset, InfoRole, TargetRole
from hypex.utils import BackendsEnum

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "segment": InfoRole(str),
        "revenue": TargetRole(float),
        "visits": InfoRole(int),
    },
    data={
        "data": {
            "user_id": list(range(12)),
            "segment": ["new", "returning"] * 6,
            "revenue": [10.0 + i for i in range(12)],
            "visits": [i % 4 for i in range(12)],
        }
    },
)
with patch.multiple(DatasetConfig, DISPLAY_ROWS=2, DISPLAY_COLS=1):
    display(data)

print("Full shape:", data.shape)
print(
    "Restored display settings:", DatasetConfig.DISPLAY_ROWS, DatasetConfig.DISPLAY_COLS
)

,user_id,...,visits
0,0,...,0
1,1,...,1
...,...,...,...
10,10,...,2
11,11,...,3


Full shape: (12, 4)
Restored display settings: 5 10


The preview shows two rows at each end and the first and last columns. The table still has shape `(12, 4)`, and exiting the block restores the original display settings.

## Limit collection of a Spark Dataset to the driver

Start a local session and move the 12 rows to Spark. Create a temporary checkpoint directory for the matching example. Set `PYSPARK_PYTHON` if it is unset so Spark workers can use the Jupyter environment.

In [2]:
import os
import sys
from tempfile import TemporaryDirectory

from pyspark.sql import SparkSession

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
checkpoints = TemporaryDirectory(prefix="hypex-config-")
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx configuration")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir(checkpoints.name)

spark_data = data.to_backend(BackendsEnum.spark, session=spark)
print("Backend:", spark_data.backend_type.value)

Backend: spark


`Backend: spark` confirms that the dataset is distributed. Set `SPARK_PANDAS_CONVERSION_LIMIT=4` and call `to_backend(BackendsEnum.pandas)`. Then allow exactly 12 rows to check whether the limit is inclusive.

In [3]:
with patch.multiple(DatasetConfig, SPARK_PANDAS_CONVERSION_LIMIT=4):
    try:
        spark_data.to_backend(BackendsEnum.pandas)
    except ValueError as error:
        if "exceed limit 4" not in str(error):
            raise
        print(error)

with patch.multiple(DatasetConfig, SPARK_PANDAS_CONVERSION_LIMIT=len(data)):
    local = spark_data.to_backend(BackendsEnum.pandas)
    print(f"Converted {len(local)} rows to {local.backend_type.value}.")
    print("Target columns:", local.search_columns(TargetRole()))

print("Restored row limit:", DatasetConfig.SPARK_PANDAS_CONVERSION_LIMIT)

to_backend conversion: 12 rows exceed limit 4
Converted 12 rows to pandas.
Target columns: ['revenue']
Restored row limit: 100000


The limit of 4 rejects the collection. At 12, the full dataset converts to pandas and keeps the outcome's role. The threshold counts rows. The [compression guide](spark-to-pandas-compression.ipynb) shows how to measure memory and compress a Spark DataFrame before collection.

## Temporarily enable exact neighbor search

For Spark matching, `MatchingConfig.FAISS_FIT_MODE="full"` enables exact search using flat indexes on the partitions. Set it around `Matching.execute()`. This example focuses on the full effect report, so `quality_tests=[]` disables the separate balance report. This argument controls diagnostics independently of the search mode.

In the simulated data, assignment depends on `X` and the true effect is 5.

In [4]:
import numpy as np
import pandas as pd

from hypex import Matching
from hypex.dataset import FeatureRole, TreatmentRole

rng = np.random.default_rng(7)
x = rng.uniform(0, 10, size=120)
treatment = rng.binomial(1, 0.1 + 0.08 * x)
outcome = 20 + 4 * x + 5 * treatment + rng.normal(0, 0.4, size=x.size)
frame = pd.DataFrame({"Treatment": treatment, "Target": outcome, "X": x})
matching_data = Dataset(
    roles={"Treatment": TreatmentRole(), "Target": TargetRole(), "X": FeatureRole()},
    data=spark.createDataFrame(frame),
)

with patch.multiple(MatchingConfig, FAISS_FIT_MODE="full"):
    print("Search mode:", MatchingConfig.FAISS_FIT_MODE)
    matched = Matching(quality_tests=[]).execute(matching_data)
    display(matched.summary)

print("Restored search mode:", MatchingConfig.FAISS_FIT_MODE)

Search mode: full


,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,5.0608,0.1502,0.0,4.7664,5.3552
ATC,5.0142,0.1642,0.0,4.6923,5.3360
ATE,5.0352,0.1328,0.0,4.7749,5.2954


Restored search mode: shuffle


The `ATE` estimate is about 5.04. Its interval, from 4.77 to 5.30, includes the true effect of 5. The original setting, `shuffle`, was restored after the block. In that mode, `FAISS_N_PROBES` controls how many clusters each query searches. See [MatchingConfig](../../../hypex/config/config.py) for the available Spark search settings.

Stop the local session and remove its temporary directory.

In [5]:
spark.stop()
checkpoints.cleanup()
print("Spark session stopped")

Spark session stopped
